# 01. Event snapshot tables (shots & passes)

PFF event JSON already contains, for every possession event:
- all player coordinates at the event frame (`homePlayers` / `awayPlayers`), with `playerId`, `speed`, `visibility`
- attacking direction (`stadiumMetadata.teamAttackingDirection`)

So no tracking (bz2) parsing is needed at this stage.

**Outputs** (`outputs/`)
- `events.csv.gz` : one row per shot / pass, with the better-option label
- `players.csv.gz` : one row per player per event (metres, attacking team always towards +x)

In [1]:
import glob
import json
import os

import pandas as pd

DATA_DIR = os.path.join("..", "data", "World_cup")
OUT_DIR = os.path.join("..", "outputs")
EVENT_TYPES = ("SH", "PA")
ACTOR_FIELD = {"SH": "shooterPlayerId", "PA": "passerPlayerId"}

files = sorted(glob.glob(os.path.join(DATA_DIR, "Event Data", "*_event.json")))
len(files)

64

## 1. One game → event rows + player rows

- `label` = `betterOptionType` is not null (analyst charted a better option)
- `non_event` = event disallowed after the fact (offside etc.). Kept, because the decision still happened.
- If the team in possession attacks to the left (`L`), x and y are flipped.

In [ ]:
def build_game(path):
    ev_rows, pl_rows = [], []
    for e in json.load(open(path, encoding="utf-8")):
        pe = e.get("possessionEvents") or {}
        ge = e.get("gameEvents") or {}
        etype = pe.get("possessionEventType")
        if etype not in EVENT_TYPES:
            continue
        sm = e.get("stadiumMetadata") or {}
        flip = sm.get("teamAttackingDirection") == "L"
        atk_home = ge.get("homeTeam") is True
        actor = pe.get(ACTOR_FIELD[etype]) or ge.get("playerId")
        bo_player = pe.get("betterOptionPlayerId")
        key = dict(game_id=e["gameId"], pe_id=e["possessionEventId"])

        n_players = n_visible = 0
        for side, atk in (("homePlayers", atk_home), ("awayPlayers", not atk_home)):
            for p in e.get(side) or []:
                if p.get("x") is None:
                    continue
                x, y = (-p["x"], -p["y"]) if flip else (p["x"], p["y"])
                vis = p.get("visibility") == "VISIBLE"
                n_players += 1
                n_visible += vis
                pl_rows.append(dict(
                    key, player_id=p.get("playerId"), atk=atk, x=x, y=y,
                    speed=p.get("speed"), visible=vis,
                    position=p.get("positionGroupType"),
                    is_actor=p.get("playerId") == actor,
                    is_bo_target=bo_player is not None and p.get("playerId") == bo_player,
                ))

        ball = (e.get("ball") or [{}])[0] or {}
        bx, by = ball.get("x"), ball.get("y")
        if flip and bx is not None:
            bx, by = -bx, -by
        ev_rows.append(dict(
            key, event_type=etype, period=ge.get("period"), time=e.get("startTime"),
            sequence=e.get("sequence"), team_id=ge.get("teamId"), atk_home=atk_home,
            non_event=bool(pe.get("nonEvent")),
            actor_id=actor, setpiece=ge.get("setpieceType"),
            pressure=pe.get("pressureType"), ball_x=bx, ball_y=by,
            body_type=pe.get("bodyType"), shot_outcome=pe.get("shotOutcomeType"),
            target_id=pe.get("targetPlayerId"), receiver_id=pe.get("receiverPlayerId"),
            pass_outcome=pe.get("passOutcomeType"),
            pressure_player_id=pe.get("pressurePlayerId"),
            label=pe.get("betterOptionType") is not None,
            bo_type=pe.get("betterOptionType"), bo_player_id=bo_player,
            n_players=n_players, n_visible=n_visible,
        ))
    return ev_rows, pl_rows


# quick look at one game
a, b = build_game(files[0])
pd.DataFrame(a).head()

## 2. All 64 games

In [3]:
ev, pl = [], []
for f in files:
    a, b = build_game(f)
    ev += a
    pl += b
ev, pl = pd.DataFrame(ev), pd.DataFrame(pl)
print(f"events={len(ev):,}  player rows={len(pl):,}")

events=68,274  player rows=1,501,943


## 3. Sanity checks

- tagged counts must match the raw JSON (SH 293, PA 324)
- almost every event should have its actor in the snapshot
- shots should sit near +x (attacking goal at x ≈ +52.5)

In [4]:
assert ev.query("event_type=='SH'").label.sum() == 293
assert ev.query("event_type=='PA'").label.sum() == 324

actor_keys = pl[pl.is_actor].set_index(["game_id", "pe_id"]).index
print("events without actor:", (~ev.set_index(["game_id", "pe_id"]).index.isin(actor_keys)).sum())

shooters = pl[pl.is_actor].merge(ev.query("event_type=='SH'")[["game_id", "pe_id"]])
print("shooter x median:", shooters.x.median())

ev.groupby(["event_type", "non_event"]).label.agg(["size", "sum"])

events without actor: 18
shooter x median: 39.316


size  sum
event_type non_event            
PA         False      66185  323
           True         530    1
SH         False       1470  290
           True          89    3

In [5]:
# better-option type distribution and target visibility
display(ev[ev.label].groupby(["event_type", "bo_type"]).size().unstack(0))
t = pl[pl.is_bo_target]
print(f"named target in snapshot: {len(t)}, visible: {t.visible.mean():.1%}")

event_type,PA,SH
bo_type,,
B,14.0,4.0
C,6.0,72.0
H,54.0,29.0
L,30.0,NaN
O,9.0,3.0
P,205.0,183.0
S,6.0,2.0


named target in snapshot: 468, visible: 87.6%


## 4. Save

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
ev.to_csv(os.path.join(OUT_DIR, "events.csv.gz"), index=False)
pl.to_csv(os.path.join(OUT_DIR, "players.csv.gz"), index=False)
print("saved to", OUT_DIR)

saved to C:\dev\football\outputs
